# Imports

In [1]:
import optuna
import pickle

import numpy as np
import pandas as pd

from tqdm import tqdm

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

/home/junior/Documentos/GitHub/kaggle-competition-predicting-student-health/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Utils

In [2]:
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        return pickle.load(file)

# Loading Datasets

In [3]:
X_train = pd.read_parquet('../data/processed/X_train_stacking_layer_one_model_60_trials.parquet')
y_train = pd.read_parquet('../data/interim/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking_layer_one_model_60_trials.parquet')

In [4]:
X_train.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.004922,0.001773,0.993304,0.004134,0.002593,0.993273
1,0.997228,0.000463,0.002309,0.995591,0.000405,0.004004
2,0.002369,0.000604,0.997026,0.001958,0.000503,0.997538
3,0.004656,0.003362,0.991982,0.002466,0.002391,0.995142
4,0.995353,0.000016,0.004632,0.994178,0.000003,0.005819


In [5]:
X_test.head()

,lgbm_0,lgbm_1,lgbm_2,xgb_0,xgb_1,xgb_2
0,0.012452,0.003199,0.984349,0.011048,0.004491,0.984461
1,0.472116,0.002331,0.525553,0.499186,0.001896,0.498918
2,0.998690,0.001012,0.000298,0.997826,0.001594,0.000579
3,0.980854,0.000018,0.019128,0.987434,0.000009,0.012557
4,0.006765,0.001847,0.991388,0.006876,0.001972,0.991152


# Machine Learning

In [6]:
label_encoder = load_pickle('../artifacts/label_encoder.pkl')

In [7]:
cols_use = ['lgbm_0', 'lgbm_1', 'lgbm_2']

In [8]:
def objective(trial, X, y):

    w0 = trial.suggest_float('weight_class_0', 0.0, 10.0, log=False)
    w1 = trial.suggest_float('weight_class_1', 0.0, 10.0, log=False)
    w2 = trial.suggest_float('weight_class_2', 0.0, 10.0, log=False)
    weights = np.array([w0, w1, w2])

    proba = X.loc[:, cols_use].to_numpy()
    
    weighted_probas = proba * weights
    pred = np.argmax(weighted_probas, axis=1)
    
    score = balanced_accuracy_score(y, pred)

    return score


study = optuna.create_study(
    direction="maximize", 
    sampler=optuna.samplers.TPESampler(seed=42), 
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=2)
)

study.optimize(
    lambda trial: objective(trial, X_train, y_train.health_condition), 
    n_trials=1000, 
    n_jobs=-1, 
    show_progress_bar=True
)

print("\nBest trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-07-05 15:04:15,093] A new study created in memory with name: no-name-ff6e9dd6-4fe5-4e64-a47a-5814f95751ba
                                                                                                                                                                                                                   

[I 2026-07-05 15:04:15,257] Trial 8 finished with value: 0.918852203716967 and parameters: {'weight_class_0': 0.3814428378181878, 'weight_class_1': 0.39411919330551215, 'weight_class_2': 3.439349748982803}. Best is trial 8 with value: 0.918852203716967.
[I 2026-07-05 15:04:15,273] Trial 1 finished with value: 0.8541214198552605 and parameters: {'weight_class_0': 6.489242893387227, 'weight_class_1': 1.2243729189974817, 'weight_class_2': 7.751264470212734}. Best is trial 8 with value: 0.918852203716967.
[I 2026-07-05 15:04:15,275] Trial 5 finished with value: 0.8400025005253053 and parameters: {'weight_class_0': 4.172370295868767, 'weight_class_1': 1.308952215170579, 'weight_class_2': 1.6014568741014656}. Best is trial 8 with value: 0.918852203716967.
[I 2026-07-05 15:04:15,276] Trial 11 finished with value: 0.8413032521260743 and parameters: {'weight_class_0': 6.072748681784892, 'weight_class_1': 2.598935670213993, 'weight_class_2': 1.0451972104993912}. Best is trial 8 with value: 0.918

Best trial: 14. Best value: 0.949616:   2%|██▉                                                                                                                                   | 22/1000 [00:00<00:21, 45.92it/s]

[I 2026-07-05 15:04:15,419] Trial 12 finished with value: 0.8675662578778782 and parameters: {'weight_class_0': 8.895016442189174, 'weight_class_1': 5.754974154026845, 'weight_class_2': 8.080092868740795}. Best is trial 4 with value: 0.9437059633423232.
[I 2026-07-05 15:04:15,421] Trial 13 finished with value: 0.8774642825630953 and parameters: {'weight_class_0': 9.208814926022495, 'weight_class_1': 7.348200751975093, 'weight_class_2': 9.964256735918514}. Best is trial 4 with value: 0.9437059633423232.
[I 2026-07-05 15:04:15,430] Trial 14 finished with value: 0.949616096912061 and parameters: {'weight_class_0': 0.4671959043557937, 'weight_class_1': 5.5882527153470605, 'weight_class_2': 5.791298450230845}. Best is trial 14 with value: 0.949616096912061.
[I 2026-07-05 15:04:15,462] Trial 16 finished with value: 0.9489327990058407 and parameters: {'weight_class_0': 0.25421834987736636, 'weight_class_1': 5.010253954794902, 'weight_class_2': 5.530522603692354}. Best is trial 14 with value: 

Best trial: 14. Best value: 0.949616:   3%|████▌                                                                                                                                 | 34/1000 [00:00<00:19, 48.93it/s]

...

[I 2026-07-05 15:04:35,390] Trial 970 finished with value: 0.9495297173056834 and parameters: {'weight_class_0': 0.38622946893422627, 'weight_class_1': 8.182395315790743, 'weight_class_2': 5.9969106736789834}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,412] Trial 972 finished with value: 0.9488406747063535 and parameters: {'weight_class_0': 0.3410616524339806, 'weight_class_1': 9.629786185826331, 'weight_class_2': 6.605335544570468}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,414] Trial 971 finished with value: 0.9494981788929518 and parameters: {'weight_class_0': 0.3673934590520931, 'weight_class_1': 6.780015814786509, 'weight_class_2': 6.0176451674293565}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,444] Trial 973 finished with value: 0.9491131715775047 and parameters: {'weight_class_0': 0.3626788734874331, 'weight_class_1': 9.54420713663942, 'weight_class_2': 5.981493029135231}. Best is trial 33

Best trial: 330. Best value: 0.94987:  99%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▍ | 988/1000 [00:20<00:00, 41.78it/s]

[I 2026-07-05 15:04:35,604] Trial 980 finished with value: 0.9497118647250854 and parameters: {'weight_class_0': 0.49321437245491234, 'weight_class_1': 8.746445560056655, 'weight_class_2': 6.612587038912376}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,608] Trial 979 finished with value: 0.9497135271080751 and parameters: {'weight_class_0': 0.4779733449237096, 'weight_class_1': 9.347453241516831, 'weight_class_2': 5.6415676662646845}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,644] Trial 981 finished with value: 0.9476069259385841 and parameters: {'weight_class_0': 0.5954658830141337, 'weight_class_1': 3.1235234680757937, 'weight_class_2': 6.560159284831501}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,665] Trial 982 finished with value: 0.9498016855660034 and parameters: {'weight_class_0': 0.5802485510738203, 'weight_class_1': 9.371620978611976, 'weight_class_2': 6.584623029720483}. Best is trial 3

Best trial: 330. Best value: 0.94987: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:20<00:00, 47.82it/s]

[I 2026-07-05 15:04:35,865] Trial 989 finished with value: 0.9496442747681032 and parameters: {'weight_class_0': 0.6079234011102213, 'weight_class_1': 7.7507057883761945, 'weight_class_2': 5.7296691927118735}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,867] Trial 990 finished with value: 0.949688535768539 and parameters: {'weight_class_0': 0.5894837303180845, 'weight_class_1': 8.517315882706425, 'weight_class_2': 5.652187637631107}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,890] Trial 991 finished with value: 0.9441303514203542 and parameters: {'weight_class_0': 0.13846818767363056, 'weight_class_1': 8.76031151560454, 'weight_class_2': 5.677886932677937}. Best is trial 330 with value: 0.9498700288669591.
[I 2026-07-05 15:04:35,928] Trial 992 finished with value: 0.9492873340836984 and parameters: {'weight_class_0': 0.6261300964997237, 'weight_class_1': 5.30610660236197, 'weight_class_2': 5.638525317074819}. Best is trial 330 

In [9]:
weights = np.array([study.best_params['weight_class_0'], study.best_params['weight_class_1'], study.best_params['weight_class_2']])
weighted_probas = X_test.loc[:, cols_use].to_numpy() * weights

pred = np.argmax(weighted_probas, axis=1)

In [10]:
sub_labels = label_encoder.inverse_transform(pred)

# Submission

In [11]:
submission = pd.read_csv('../data/submission/sample_submission.csv')
submission['health_condition'] = sub_labels

submission.to_csv('../data/submission/submission_from_5.1.1_lightgbm_submission.csv', index=False)

In [12]:
submission.head()

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,at-risk
3,690091,at-risk
4,690092,unhealthy
